# **Explore Hessen Soccer**

In [1]:
from utils import write_scraped_df, concat_scrape
import geopandas as gpd
import pandas as pd
import osmnx as ox
pd.set_option('display.max_rows', 200)

# define coordinate reference system
CRS = 5243

See ``wiki_scraper`` from utils to see how each table was scraped.

In [2]:
liga_dict = {
'verbandsliga_nord': 'https://www.fussball.de/spieltag/vl-grnord-hessen-verbandsliga-herren-saison2627-hessen/-/staffel/0318HV9REO00000AVS5489BUVV628VP4-G#!/section/matches',
'verbandsliga_mitte': 'https://www.fussball.de/spieltag/vl-grmitte-hessen-verbandsliga-herren-saison2627-hessen/-/staffel/0318I3K77C00000BVS5489BUVV628VP4-G#!/section/matches',
'verbandsliga_sued': 'https://www.fussball.de/spieltag/vl-grsued-hessen-verbandsliga-herren-saison2627-hessen/-/staffel/0318I2SMPG00000CVS5489BUVV628VP4-G#!/section/matches',
'hessenliga': 'https://www.fussball.de/spieltag/lotto-hessenliga-hessen-hessenliga-herren-saison2627-hessen/-/staffel/0318HOM0R000000BVS5489BUVV628VP4-G#!/section/matches',
'regionalliga_suedwest': 'https://www.fussball.de/spieltag/regionalliga-suedwest-deutschland-regionalliga-suedwest-herren-saison2627-deutschland/-/staffel/031D0NRQ3O000004VS5489BUVUR5FS5A-G#!/section/matches',
'3_bundesliga': 'https://www.fussball.de/spieltag/3liga-deutschland-3liga-herren-saison2627-deutschland/-/staffel/031BH6P20G00000HVS5489BUVUR5FS5A-G#!/section/matches',
'2_bundesliga': 'https://www.fussball.de/spieltag/2bundesliga-deutschland-2bundesliga-herren-saison2627-deutschland/-/staffel/031BG5ESP800000TVS5489BUVUR5FS5A-G#!/section/matches',
'bundesliga': 'https://www.fussball.de/spieltag/bundesliga-deutschland-bundesliga-herren-saison2627-deutschland/-/staffel/031BFREJC800001BVS5489BUVUR5FS5A-G#!/section/matches'
}

# write scraped tables from URL
write_scraped_df(d=liga_dict)
bl_to_vl = concat_scrape(liga_dict)
bl_to_vl.to_csv('bl_to_vl.csv', index=False)
bl_to_vl.info()

<class 'pandas.DataFrame'>
Index: 143 entries, 0 to 17
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   pl.platz            0 non-null      float64
 1   pl.platz.1          143 non-null    float64
 2   mannschaft          143 non-null    str    
 3   sp.spiele           143 non-null    int64  
 4   g                   143 non-null    int64  
 5   u                   143 non-null    int64  
 6   v                   143 non-null    int64  
 7   torv.torverhältnis  143 non-null    str    
 8   tordifferenz        143 non-null    int64  
 9   pkt.punkte          143 non-null    int64  
 10  liga                143 non-null    str    
dtypes: float64(2), int64(6), str(3)
memory usage: 13.4 KB


In [3]:
bl_to_vl

,pl.platz,pl.platz.1,mannschaft,sp.spiele,g,u,v,torv.torverhältnis,tordifferenz,pkt.punkte,liga
0,NaN,1.0,SG Klei./​Hun./​Doh.,18,12,2,4,42 : 26,16,38,verbandsliga_nord
1,NaN,2.0,SV Steinbach,17,11,3,3,45 : 22,23,36,verbandsliga_nord
2,NaN,3.0,Tuspo Rengershausen,18,10,5,3,52 : 28,24,35,verbandsliga_nord
3,NaN,4.0,SV Weidenhausen,17,11,1,5,26 : 19,7,34,verbandsliga_nord
4,NaN,5.0,SG Bad Soden,18,11,1,6,47 : 22,25,33,verbandsliga_nord
5,NaN,6.0,FSV Wolfhagen,18,10,2,6,29 : 17,12,32,verbandsliga_nord
6,NaN,7.0,OSC Vellmar,18,9,2,7,44 : 38,6,29,verbandsliga_nord
7,NaN,8.0,TSV 1900 Wabern,18,7,5,6,30 : 25,5,26,verbandsliga_nord
8,NaN,9.0,SV Buchonia Flieden,18,6,6,6,34 : 33,1,24,verbandsliga_nord
9,NaN,10.0,FSV Dörnberg,18,7,3,8,24 : 37,-13,24,verbandsliga_nord


# Geocode

In [4]:
gdf_list = []
geocode_manually = []
for i, mannschaft in enumerate(bl_to_vl['mannschaft']):
    try:
        geocoded_mannschaft = ox.geocode_to_gdf(mannschaft)
        geocoded_mannschaft['mannschaft'] = mannschaft
        # geocoded_mannschaft['liga'] = bl_to_vl.iloc[i]['liga'] # add liga w/ merge
        gdf_list.append(geocoded_mannschaft) # add the club name and league to the record for merging
    except:
        print(mannschaft)
        geocode_manually.append(mannschaft)
        continue

print('Geocoded Teams',len(gdf_list))
print('Need Geocoding',len(bl_to_vl))

SG Klei./​Hun./​Doh.
FSV Wolfhagen
OSC Vellmar
FSV Dörnberg
SG Barockstadt Fulda-Lehnerz II
SG Calden/​Meimbressen
FC Britannia Eichenzell
Tuspo Grebenstein
SG Bronnzell
Lichtenauer FV
TSV Steinbach Haiger II
SG Walluf
FV Biebrich 02
FC Ederbergland
SF/​BG Marburg
SV Zeilsheim
Türkischer SV Wiesbaden
SSC Juno Burg
SG Obbornh./​Bellersh.
TSF Heuchelheim
TuRa Niederhöchstadt
SV Pars Neu-Isenburg
DJK Spfr. Bad Homburg
SV Bosnien/​Herzegowina Ffm
Spfrd. Seligenstadt
VfR Fehlheim
SG Bornheim/​GW Ffm
SV Rot-Weiss Walldorf II
Kickers Offenbach II
SG Rot-Weiss Ffm.
1. Hanauer FC
Spvgg.Neu-Isenburg
TSV Eintr. Stadtallendorf
Türk Gücü Friedberg
SV Darmstadt 98 U21
FC Bayern Alzenau
FC Turabdin-Babylon
FSV Fernwald
Hünfelder SV
SC 1960 Hanau
SV Rot-Weiss Walldorf
SV Hummetroth
KSV Hessen Kassel II
SGV Heilbronn-Freiberg
VfR Mannheim
SV Stuttgarter Kickers
Kickers Offenbach
TSV Steinbach Haiger
SC Freiburg II
Eintracht Frankfurt II
SSV Ulm 1846 Fußball
1. FSV Mainz 05 II
1. FC Kaiserslautern II
MS

In [ ]:
geocoded = pd.concat(objs=[gdf_list])
manually_geocoded_df = gpd.read_file('new_manually_geocoded.csv')
missing_manually_geocoded_df = gpd.read_file('missing_clubs.csv')
print(len(manually_geocoded_df))
print(len(missing_manually_geocoded_df))

64
12


# Merge

In [6]:
gdf = gpd.read_file('data/hessen_fussball_gdf.geojson')

In [22]:
gdf = gdf.to_crs(CRS)
gdf.head(4)

,bbox_west,bbox_south,bbox_east,bbox_north,place_id,osm_type,osm_id,lat,lon,class,type,place_rank,importance,addresstype,name,display_name,mannschaft,geometry
0,9.6022628,50.0066357,9.6040563,50.0077706,136991363,way,26572916,50.0072173,9.6031575,leisure,pitch,30,5.472266727698203e-05,leisure,SV Steinbach,"SV Steinbach, Lohrer Straße, Steinbach, Lohr a...",SV Steinbach,POINT (9.603 50.007)
1,9.4561795,51.2516624,9.4565361,51.2518587,139962303,way,168191053,51.2517606,9.4563578,amenity,community_centre,30,5.451514457918696e-05,amenity,Vereinsheim Tuspo Rengershausen,"Vereinsheim Tuspo Rengershausen, Am Waldsportp...",Tuspo Rengershausen,POINT (9.456 51.252)
2,9.9688512,51.2085841,9.9692165,51.2087376,139534820,way,264204205,51.208656,9.9690385,club,sport,30,5.000152590214416e-05,club,SV Adler Vereinsheim,"SV Adler Vereinsheim, K 37, Weidenhausen, Meiß...",SV Weidenhausen,POINT (9.969 51.209)
3,8.4858928,50.1426272,8.4872518,50.1437036,138776756,way,23080351,50.1431635,8.4865731,leisure,pitch,30,5.398413061718192e-05,leisure,SG Bad Soden,"SG Bad Soden, Kelkheimer Straße, Bad Soden am ...",SG Bad Soden,POINT (8.487 50.143)


In [23]:
gdf.groupby('osm_type').count()

,bbox_west,bbox_south,bbox_east,bbox_north,place_id,osm_id,lat,lon,class,type,place_rank,importance,addresstype,name,display_name,mannschaft,geometry
osm_type,,,,,,,,,,,,,,,,,
relation,58,58,58,58,58,58,58,58,58,58,58,58,58,58,58,58,58
way,85,85,85,85,85,85,85,85,85,85,85,85,85,85,85,85,85


In [21]:
ox.geocode_to_gdf('w26572916', by_osmid=True)

,geometry,bbox_west,bbox_south,bbox_east,bbox_north,place_id,osm_type,osm_id,lat,lon,class,type,place_rank,importance,addresstype,name,display_name
0,"POLYGON ((9.60226 50.00727, 9.60345 50.00664, ...",9.602263,50.006636,9.604056,50.007771,136991363,way,26572916,50.007217,9.603158,leisure,pitch,30,0.000055,leisure,SV Steinbach,"SV Steinbach, Lohrer Straße, Steinbach, Lohr a..."


In [10]:
gdf.explore(tooltip=False, popup=True)

In [ ]:
# write geocoded df
# gdf.to_file('data/hessen_fussball_gdf.geojson', driver='GeoJSON')